# 04c · Failure Attribution

**Purpose.** For non-exact predictions, separate **tool failure** vs **reference gap** vs **annotation disagreement** (e.g. ORF1a/ORF1ab granularity). This is the biological-insight section reviewers like. Feeds **Figure 6**.

## Inputs

The per-prediction table from 04a (`tblastn_vs_truth.tsv`) + truth annotation profile.

In [ ]:
from pathlib import Path
import sys, time

import pandas as pd
import matplotlib.pyplot as plt

# --- anchor ROOT to the repo (folder that contains app/src) ---
ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "app" / "src").exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA   = ROOT / "app" / "data"
CONFIG = ROOT / "app" / "config"

# References (verify these are the intended ref records for the paper):
FMD_REF    = DATA / "FMD"  / "FMD_ref_test.gb"        # alt: FMD_FJ175661_Anno.gb
FMD_QUERY  = DATA / "FMD"  / "FMD_100seq_anno.gb"
PRRS_REF   = DATA / "PRRS" / "PRRS_ref_test.gb"       # alt: PRRS_MT746146_Anno.gb
PRRS_QUERY = DATA / "PRRS" / "PRRS_100seq_anno.gb"
PED_REFS   = {"ref_1": DATA / "PED" / "PED_ref_1.gb",
              "ref_2": DATA / "PED" / "PED_ref_2.gb"}
PED_QUERY  = DATA / "PED" / "PED_100seqs.gb"

# Run toggle: keep False for a fast smoke test, True for the full 100-record run.
RUN_FULL = False
SAMPLE_N = 10


In [ ]:
# outputs land inside this unit folder so figures/tables sit next to the notebook
UNIT_DIR = ROOT / "app" / "validation" / "08_failure_attribution"
OUT = UNIT_DIR / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
print("outputs ->", OUT)

## Load — failed / non-exact cases from 04a

In [ ]:
ACC = ROOT / "app" / "validation" / "02_lifting_accuracy" / "outputs"
per_pred = pd.concat([pd.read_csv(f, sep="\t") for f in ACC.glob("*/tblastn_vs_truth.tsv")],
                     ignore_index=True)
non_exact = per_pred[~per_pred["exact_match"].fillna(False)].copy()
print(len(non_exact), "non-exact cases")
non_exact.head()

## Metrics — attribute each failure

> ⚠️ **TODO**: define the attribution rule per case: `tool_failure` (missing/low-cov/invalid boundary), `ref_annotation_gap` (truth lacks the gene the ref expects), `annotation_disagreement` (granularity/name-convention mismatch, e.g. ORF1ab vs ORF1a/1b). Encode as a function of the failure_status + truth profile columns.

In [ ]:
def attribute(row):
    status = str(row.get("status", "")).lower()
    if "granularity" in status or "orf1ab" in str(row.get("pred_name", "")).lower():
        return "annotation_disagreement"
    if "missing" in status or "no_hit" in status:
        return "ref_annotation_gap"      # TODO: refine vs tool_failure using coverage/identity
    return "tool_failure"

non_exact["attribution"] = non_exact.apply(attribute, axis=1)
attr = non_exact.groupby(["virus", "attribution"]).size().reset_index(name="n")
attr.to_csv(OUT / "failure_attribution.tsv", sep="\t", index=False)
attr

## Figure

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
attr.pivot_table(index="virus", columns="attribution", values="n", fill_value=0).plot.bar(stacked=True, ax=ax)
ax.set_ylabel("cases"); ax.set_title("Failure attribution")
fig.tight_layout(); fig.savefig(OUT / "failure_attribution.png", dpi=200)

## Interpretation

> ⚠️ **TODO**: phần lớn 'fail' là annotation disagreement chứ không phải tool sai — luận điểm mạnh cho Discussion.